# Life2Lang — Tokenise Instruction Dataset

Loads `khairi/life2lang-instruction-dataset`, formats and tokenises every
split, then pushes the result to HF Hub as
`khairi/life2lang-instruction-dataset-tokenized`.

Each example is formatted as:
```
input_text  = "{prefix}\n{input}"
target_text = "{target}"
```
then tokenised with `padding='max_length'` and `max_length=1024`.
The saved dataset keeps only `input_ids`, `attention_mask`, and `labels`.

## 1 · Install

In [ ]:
!pip install -q git+https://github.com/abidikhairi/life2lang.git

## 2 · Authenticate

In [1]:
from huggingface_hub import login
login()  # needs write access to push the tokenized dataset

## 3 · Configuration

In [4]:
DATASET_ID     = "khairi/life2lang-instruction-dataset"
BASE_MODEL     = "khairi/life2lang-base"
MAX_LENGTH     = 1024
MAP_BATCH_SIZE = 512

# Derived — keeps the naming convention {base_name}-tokenized
HUB_DATASET_ID = DATASET_ID + "-tokenized"
print(f"Output dataset : {HUB_DATASET_ID}")

Output dataset : khairi/life2lang-instruction-dataset-tokenized


## 4 · Load dataset

In [3]:
from datasets import load_dataset

dataset   = load_dataset(DATASET_ID)
train_raw = dataset["train"]
valid_raw = dataset["test"]

print(f"Train size : {len(train_raw):,}")
print(f"Valid size : {len(valid_raw):,}")
print(f"Columns    : {train_raw.column_names}")
train_raw[0]

train.parquet:   0%|          | 0.00/907M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/999k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2788841 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3145 [00:00<?, ? examples/s]

Train size : 2,788,841
Valid size : 3,145
Columns    : ['entry', 'prefix', 'input', 'target', 'aspect', 'type', 'source']


{'entry': 'Q6GM16',
 'prefix': 'predict biological processes from protein sequence:',
 'input': '<protein> M S G P G L L V E L L G E K L V N S E R E E A D V Q A L G S R V S L I G L L F G C G M S A P C L Q L L P G L K D F Y C K T R D R L E I V F V S S D P D Q K K W Q L F V K D M P W L A L P Y Q E K H R K L K L W N K F R I S N I P S L I F I E A S T V K T V C R N G L L L V K D D P E G L E F P W G P K P F C E V I A G P L I R N N S Q S Q E S S T L E G S Y V G I Y F S A Y W C P P C R S L T R V L V E S Y R K I K E S G Q K F E I V L V S A D R S E E S F K Q Y F S E M P W L A V P Y S D E A R R S R L N R L Y G I Q G I P N L I I L D P K G E V I T R Q G R V E V L R D I D C K E F P W H P K P V V E L T E L N A V Q L N E G P C L V L F V D S E D E G E S E A A K Q L I Q P I A E K I I A Q H K A K D E D A P L L F F V A G E D D M T D S L R D F T N L P E A A P L L T I L D M S A R A K Y V M D V E E I T P E I V Q S F V T D F L A E K L K P E P I </protein>',
 'target': 'cell differentiation',
 'aspect': 'GO-BP

## 5 · Tokeniser

In [5]:
from life2lang.models import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {tokenizer.vocab_size}")

[ERROR] `head_mask` is part of T5Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/khairi/work/phd/projects/life2lang/src/life2lang/models/t5/modeling_t5.py.
[ERROR] `cache_position` is part of T5Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/khairi/work/phd/projects/life2lang/src/life2lang/models/t5/modeling_t5.py.
[ERROR] `head_mask` is part of T5ForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/khairi/work/phd/projects/life2lang/src/life2lang/models/t5/modeling_t5.py.
[ERROR] `cache_position` is part of T5ForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/khairi/work/phd/projects/life2lang/src/life2lang/models/t5/modeling_t5.py.
[ERROR] `head_mask` is part of T5EncoderModel.forward's signature, but not documente

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

Vocab size : 32000


## 6 · Format & tokenise

In [ ]:
def format_batch(batch):
    input_texts  = [f"{p}\n{i}" for p, i in zip(batch["prefix"], batch["input"])]
    target_texts = batch["target"]

    return {
        "input_text":  input_texts,
        "target_text": target_texts,
    }


def tokenize_batch(batch):
    input_texts  = tokenizer(batch["input_text"],  padding="max_length", max_length=MAX_LENGTH, truncation=True)
    target_texts = tokenizer(batch["target_text"], padding="max_length", max_length=MAX_LENGTH, truncation=True)

    return {
        "input_ids":      input_texts["input_ids"],
        "attention_mask": input_texts["attention_mask"],
        "labels":         target_texts["input_ids"],
    }


keep_cols = ["input_ids", "attention_mask", "labels"]

train_data = (
    train_raw
    .map(format_batch,   batched=True, batch_size=MAP_BATCH_SIZE)
    .map(tokenize_batch, batched=True, batch_size=MAP_BATCH_SIZE)
    .select_columns(keep_cols)
)

valid_data = (
    valid_raw
    .map(format_batch,   batched=True, batch_size=MAP_BATCH_SIZE)
    .map(tokenize_batch, batched=True, batch_size=MAP_BATCH_SIZE)
    .select_columns(keep_cols)
)

print(f"Sample input length : {len(train_data[0]['input_ids'])}")
print(f"Sample label length : {len(train_data[0]['labels'])}")

Map:   0%|          | 0/2788841 [00:00<?, ? examples/s]

Map:   0%|          | 0/2788841 [00:00<?, ? examples/s]

## 7 · Push to Hub

In [ ]:
from datasets import DatasetDict

tokenized = DatasetDict({
    "train": train_data,
    "test":  valid_data,
})

tokenized.push_to_hub(HUB_DATASET_ID)
print(f"Pushed → https://huggingface.co/datasets/{HUB_DATASET_ID}")